In [23]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Validation
# Validate Silver -> Gold consistency
# ============================================================

# ------------------------------------------------------------
# 1. Load tables
# ------------------------------------------------------------

silver_customers = spark.table("silver_customers")
silver_products = spark.table("silver_products")
silver_orders = spark.table("silver_orders")
silver_order_items = spark.table("silver_order_items")
silver_payments = spark.table("silver_payments")

dim_customer = spark.table("dim_customer")
dim_product = spark.table("dim_product")
dim_date = spark.table("dim_date")

fact_orders = spark.table("fact_orders")
fact_sales = spark.table("fact_sales")
fact_payments = spark.table("fact_payments")


# ============================================================
# TEST 1 — Silver vs Gold row counts
# ============================================================

print("============================================================")
print("TEST 1 — ROW COUNTS")
print("============================================================")

counts = [
    ("silver_customers", silver_customers.count()),
    ("dim_customer", dim_customer.count()),

    ("silver_products", silver_products.count()),
    ("dim_product", dim_product.count()),

    ("silver_orders", silver_orders.count()),
    ("fact_orders", fact_orders.count()),

    ("silver_order_items", silver_order_items.count()),
    ("fact_sales", fact_sales.count()),

    ("silver_payments", silver_payments.count()),
    ("fact_payments", fact_payments.count())
]

for table_name, count in counts:
    print(f"{table_name:25} : {count}")


# ============================================================
# TEST 2 — Duplicate primary/business keys
# ============================================================

print("\n============================================================")
print("TEST 2 — DUPLICATES")
print("============================================================")

duplicate_tests = {
    "silver_customers": ("customer_id", silver_customers),
    "dim_customer": ("customer_id", dim_customer),

    "silver_products": ("product_id", silver_products),
    "dim_product": ("product_id", dim_product),

    "silver_orders": ("order_id", silver_orders),
    "fact_orders": ("order_id", fact_orders),

    "silver_order_items": ("order_item_id", silver_order_items),
    "fact_sales": ("order_item_id", fact_sales),

    "silver_payments": ("payment_id", silver_payments),
    "fact_payments": ("payment_id", fact_payments)
}

for table_name, (key_column, df) in duplicate_tests.items():

    duplicate_count = (
        df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name:25} | "
        f"key={key_column:15} | "
        f"duplicate keys={duplicate_count}"
    )


# ============================================================
# TEST 3 — Null surrogate keys
# ============================================================

print("\n============================================================")
print("TEST 3 — NULL SURROGATE KEYS")
print("============================================================")

null_customer_keys = (
    fact_orders
    .filter(F.col("customer_key").isNull())
    .count()
)

null_sales_customer_keys = (
    fact_sales
    .filter(F.col("customer_key").isNull())
    .count()
)

null_product_keys = (
    fact_sales
    .filter(F.col("product_key").isNull())
    .count()
)

null_payment_customer_keys = (
    fact_payments
    .filter(F.col("customer_key").isNull())
    .count()
)

null_order_dates = (
    fact_orders
    .filter(F.col("date_key").isNull())
    .count()
)

null_sales_dates = (
    fact_sales
    .filter(F.col("date_key").isNull())
    .count()
)

null_payment_dates = (
    fact_payments
    .filter(F.col("date_key").isNull())
    .count()
)

print(f"fact_orders   - null customer_key : {null_customer_keys}")
print(f"fact_orders   - null date_key     : {null_order_dates}")

print(f"fact_sales    - null customer_key : {null_sales_customer_keys}")
print(f"fact_sales    - null product_key  : {null_product_keys}")
print(f"fact_sales    - null date_key     : {null_sales_dates}")

print(f"fact_payments - null customer_key : {null_payment_customer_keys}")
print(f"fact_payments - null date_key     : {null_payment_dates}")


# ============================================================
# TEST 4 — Orphan customer keys
# ============================================================

print("\n============================================================")
print("TEST 4 — ORPHAN CUSTOMER KEYS")
print("============================================================")

orphan_order_customers = (
    fact_orders
    .join(
        dim_customer.select("customer_key"),
        on="customer_key",
        how="left_anti"
    )
    .count()
)

orphan_sales_customers = (
    fact_sales
    .join(
        dim_customer.select("customer_key"),
        on="customer_key",
        how="left_anti"
    )
    .count()
)

orphan_payment_customers = (
    fact_payments
    .join(
        dim_customer.select("customer_key"),
        on="customer_key",
        how="left_anti"
    )
    .count()
)

print(f"fact_orders   - orphan customers : {orphan_order_customers}")
print(f"fact_sales    - orphan customers : {orphan_sales_customers}")
print(f"fact_payments - orphan customers : {orphan_payment_customers}")


# ============================================================
# TEST 5 — Orphan product keys
# ============================================================

print("\n============================================================")
print("TEST 5 — ORPHAN PRODUCT KEYS")
print("============================================================")

orphan_products = (
    fact_sales
    .join(
        dim_product.select("product_key"),
        on="product_key",
        how="left_anti"
    )
    .count()
)

print(f"fact_sales - orphan products : {orphan_products}")


# ============================================================
# TEST 6 — Order totals vs sales totals
# ============================================================

print("\n============================================================")
print("TEST 6 — ORDER TOTAL RECONCILIATION")
print("============================================================")

order_totals = (
    fact_orders
    .groupBy("order_id")
    .agg(
        F.max("total_amount").alias("order_total")
    )
)

sales_totals = (
    fact_sales
    .groupBy("order_id")
    .agg(
        F.round(F.sum("sales_amount"), 2).alias("sales_total")
    )
)

reconciliation = (
    order_totals
    .join(
        sales_totals,
        on="order_id",
        how="inner"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("order_total") - F.col("sales_total"),
            2
        )
    )
)

mismatched_orders = (
    reconciliation
    .filter(F.abs(F.col("difference")) > 0.01)
    .count()
)

print(f"Orders with sales mismatch : {mismatched_orders}")

reconciliation.orderBy(
    F.abs(F.col("difference")).desc()
).show(10, truncate=False)


# ============================================================
# TEST 7 — Payment totals
# ============================================================

print("\n============================================================")
print("TEST 7 — PAYMENT TOTALS")
print("============================================================")

payment_summary = (
    fact_payments
    .agg(
        F.count("*").alias("payment_count"),
        F.round(F.sum("amount"), 2).alias("total_payment_amount")
    )
)

payment_summary.show()


# ============================================================
# TEST 8 — Fact table summaries
# ============================================================

print("\n============================================================")
print("TEST 8 — GOLD FACT SUMMARIES")
print("============================================================")

print("FACT ORDERS")
fact_orders.select(
    F.count("*").alias("orders"),
    F.round(F.sum("total_amount"), 2).alias("order_value")
).show()

print("FACT SALES")
fact_sales.select(
    F.count("*").alias("order_items"),
    F.sum("quantity").alias("units_sold"),
    F.round(F.sum("sales_amount"), 2).alias("sales_value")
).show()

print("FACT PAYMENTS")
fact_payments.select(
    F.count("*").alias("payments"),
    F.round(F.sum("amount"), 2).alias("payment_value")
).show()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n============================================================")
print("GOLD VALIDATION COMPLETE")
print("============================================================")

print(f"Duplicate customer keys       : {duplicate_tests['dim_customer'][1].groupBy('customer_id').count().filter(F.col('count') > 1).count()}")
print(f"Duplicate product keys        : {duplicate_tests['dim_product'][1].groupBy('product_id').count().filter(F.col('count') > 1).count()}")
print(f"Duplicate order keys          : {duplicate_tests['fact_orders'][1].groupBy('order_id').count().filter(F.col('count') > 1).count()}")
print(f"Duplicate sales keys          : {duplicate_tests['fact_sales'][1].groupBy('order_item_id').count().filter(F.col('count') > 1).count()}")
print(f"Duplicate payment keys        : {duplicate_tests['fact_payments'][1].groupBy('payment_id').count().filter(F.col('count') > 1).count()}")

print(f"Orphan customer keys          : {orphan_order_customers + orphan_sales_customers + orphan_payment_customers}")
print(f"Orphan product keys            : {orphan_products}")
print(f"Order/sales mismatches         : {mismatched_orders}")

print("\nValidation notebook finished.")

StatementMeta(, 2b4bf66f-d60f-42e6-b496-a23623a8eedb, 29, Finished, Available, Finished, False)

TEST 1 — ROW COUNTS
silver_customers          : 9996
dim_customer              : 9996
silver_products           : 997
dim_product               : 997
silver_orders             : 49980
fact_orders               : 49980
silver_order_items        : 119569
fact_sales                : 119569
silver_payments           : 49978
fact_payments             : 49978

TEST 2 — DUPLICATES
silver_customers          | key=customer_id     | duplicate keys=0
dim_customer              | key=customer_id     | duplicate keys=0
silver_products           | key=product_id      | duplicate keys=0
dim_product               | key=product_id      | duplicate keys=0
silver_orders             | key=order_id        | duplicate keys=0
fact_orders               | key=order_id        | duplicate keys=0
silver_order_items        | key=order_item_id   | duplicate keys=0
fact_sales                | key=order_item_id   | duplicate keys=0
silver_payments           | key=payment_id      | duplicate keys=0
fact_payments       